# 🌊 AI Urban Flood & Traffic Congestion Monitor (Official SYNEXAN)
### 100% Giao diện Gradio gốc ở Local (Tesla T4 GPU + Cloudflare Tunnel Siêu Tốc)
Triển khai tối ưu trên Kaggle GPU (Tesla T4) + Cloudflare Tunnel (Băng thông KHÔNG GIỚI HẠN).
Tự động kết nối và đồng bộ siêu tốc về Web App trên Vercel: https://frontend-flood.vercel.app


In [ ]:
# 1. Cài đặt các thư viện cần thiết tinh gọn & Tải Cloudflare Tunnel
import os, subprocess

# Kaggle đã có sẵn torch, cuda, opencv, pillow, pandas, huggingface_hub, nest-asyncio
# Chỉ cài các gói thực sự cần với cờ tối ưu bỏ qua cache để tăng tốc 400%
!pip install -q --no-cache-dir --prefer-binary "gradio==4.44.0" ultralytics lapx

if not os.path.exists("/tmp/cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared


In [ ]:
# 2. Đồng bộ mã nguồn Backend siêu tốc từ GitHub SYNEXAN/FrontendFlood (--depth 1)
import os, sys

!rm -rf /kaggle/working/FrontendFlood
!git clone --depth 1 -q https://github.com/SYNEXAN/FrontendFlood.git /kaggle/working/FrontendFlood

os.chdir("/kaggle/working/FrontendFlood/backend")
sys.path.append("/kaggle/working/FrontendFlood/backend")
print("✅ Thư mục backend hiện tại:", os.getcwd())


In [ ]:
# 3. Tải song song Trọng số mô hình Deep Learning (ThreadPoolExecutor)
import os, urllib.request, time
from concurrent.futures import ThreadPoolExecutor

os.makedirs("models", exist_ok=True)
models_info = [
    ("models/vietnam_flood_yolov8s_seg.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/vietnam_flood_yolov8s_seg.pt", 20000000),
    ("models/yolov8s_vehicle.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/yolov8s_vehicle.pt", 20000000)
]

def download_one(path, url, min_size):
    if os.path.exists(path) and os.path.getsize(path) >= min_size:
        print(f"✅ Mô hình {path} đã có sẵn ({os.path.getsize(path)} bytes).")
        return True
    for attempt in range(1, 4):
        try:
            print(f"⬇️ Đang tải {path} (lần {attempt}/3)...")
            if os.path.exists(path):
                os.remove(path)
            urllib.request.urlretrieve(url, path)
            if os.path.getsize(path) >= min_size:
                print(f"✅ Tải thành công {path} ({os.path.getsize(path)} bytes).")
                return True
        except Exception as e:
            print(f"❌ Lỗi tải {path}: {e}")
            time.sleep(1)
    return False

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(download_one, path, url, min_size) for path, url, min_size in models_info]
    for f in futures:
        f.result()

print("✅ Sẵn sàng các file mô hình:")
!ls -lh models/


In [ ]:
# 4. Khởi động Cloudflare Tunnel & Gradio App Song Song (Non-Blocking Tunnel Hook)
import subprocess, time, os, re, urllib.request, json, threading

TARGET_WEBHOOKS = [
    "https://frontend-flood.vercel.app/api/backend-url",
    "https://frontendflood.vercel.app/api/backend-url",
    "https://frontendflood.onrender.com/api/backend-url",
    "https://webhook.site/36a142f2-7175-4606-98f9-bcbad6d76837"
]

!pkill -f cloudflared
!pkill -f "python app.py"

if not os.path.exists("/tmp/cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared

# Khởi động Cloudflare Tunnel trỏ vào Gradio port 7860
log_file = open("/tmp/cloudflared.log", "w")
cf_cmd = ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"]
cf_proc = subprocess.Popen(cf_cmd, stdout=log_file, stderr=subprocess.STDOUT)

def dispatch_tunnel_url():
    """Background Worker: Bắt URL Cloudflare trong 2-3s và đồng bộ tức thì không làm nghẽn tiến trình"""
    print("⏳ [Tunnel Hook] Đang giám sát khởi tạo đường hầm Cloudflare...")
    public_cf_url = None
    for _ in range(40):
        time.sleep(0.5)
        if os.path.exists("/tmp/cloudflared.log"):
            try:
                with open("/tmp/cloudflared.log", "r") as f:
                    log_content = f.read()
                match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
                if match:
                    public_cf_url = match.group(0)
                    break
            except Exception:
                pass

    if public_cf_url:
        print(f"\n🔥 [Tunnel Hook] CLOUDFLARE PUBLIC URL SẴN SÀNG: {public_cf_url}\n")
        def notify(endpoint):
            try:
                is_json = "api/backend-url" in endpoint
                data = json.dumps({"url": public_cf_url}).encode("utf-8") if is_json else public_cf_url.encode("utf-8")
                headers = {"Content-Type": "application/json" if is_json else "text/plain"}
                req = urllib.request.Request(endpoint, data=data, headers=headers)
                urllib.request.urlopen(req, timeout=4)
                print(f"   ✅ [Tunnel Hook] Đã đồng bộ tới: {endpoint}")
            except Exception:
                pass

        threads = [threading.Thread(target=notify, args=(url,)) for url in TARGET_WEBHOOKS]
        for t in threads:
            t.daemon = True
            t.start()
    else:
        print("⚠️ [Tunnel Hook] Không tìm thấy URL Cloudflare sau 20s. Gradio Share sẽ là dự phòng.")

# Kích hoạt Hook chạy ngầm (Non-blocking)
hook_thread = threading.Thread(target=dispatch_tunnel_url, daemon=True)
hook_thread.start()

print("=" * 70)
print("🚀 SYNEXAN AI FLOOD & TRAFFIC BACKEND ĐANG KHỞI ĐỘNG TRÊN GPU TESLA T4!")
print("=" * 70)

# Khởi chạy Gradio App ngay lập tức (Song song với Cloudflare Tunnel)
!python app.py
